In [ ]:
import pandas as pd

#Loading the DNA Dataset into a Pandas DataFrame    
filepath = 'dna_sequence_dataset.csv'
df = pd.read_csv(filepath)

#Sequence Length
df["Length"] = df["DNA_Sequence"].str.len()

#Individual occurences of A,G,T and C
for base in ["A", "T", "G", "C"]:
    df[f"Count_{base}"] = df["DNA_Sequence"].str.count(base)

#GC & AT Percent
df["GC%"] = ((df["Count_G"] + df["Count_C"])/df["Length"] * 100).round(2)
df["AT%"] = ((df["Count_A"] + df["Count_T"])/df["Length"] * 100).round(2)

#Determining the non-valid characters
df["Invalid_Chars"] = (
    df["DNA_Sequence"].str.findall(r"[^AGTCagtc]").str.join("")
    )
df["Has_Invalid"] = df["Invalid_Chars"] != ""
invalid_count = df["Has_Invalid"].sum()

#Finding user-specified pattern
user_pattern = str(input("Enter the DNA Sequence you wish to check"))
df[f"Pattern_{user_pattern}"] = df["DNA_Sequence"].str.count(user_pattern)

#Finding start and stop codons
df["Start_Codon_ATG"] = df["DNA_Sequence"].str.count("ATG")
df["Stop_Codons"] = df["DNA_Sequence"].str.count(r"TAA|TAG|TGA")

#Number of Bases Calculation
for base in ['A','T','G','C']:
    df[f"Freq_{base}"] = (df[f"Count_{base}"]/df["Length"]).round(3)
species_summary = (
    df.groupby("Species")
    .agg(
        Sample_Count=("Sample_ID", "count"),
        Avg_Length=("Length", "mean"),
        Avg_GC_Percent=("GC%", "mean"),
        Avg_AT_Percent=("AT%", "mean"),
    )
    .round(2)
)

#Highest and Lowest Percentage of GC
max_gc_percent = df["GC%"].max()
min_gc_percent = df["GC%"].min()
highest_gc_samples = df[df["GC%"] == max_gc_percent][
    ["Sample_ID", "Species", "Length", "GC%"]]
lowest_gc_samples = df[df["GC%"] == min_gc_percent][
    ["Sample_ID", "Species","Length","GC%"]]

#Summary Report
print("-- DNA Analysis Report --")
print(f"Total Samples Analysed: {len(df)}")
if invalid_count > 0:
    print("\n--- Invalid Samples Flagged ---")
    print(
        df[df["Has_Invalid"]][
            ["Sample_ID", "Species", "DNA_Sequence", "Invalid_Chars"]
        ].to_string(index=False)
    )
print(f"\nSamples containing invalid nucleotides: : {invalid_count}")
print("\nSpecies Comparison")
print(species_summary)

print(f"\nHighest GC%: {max_gc_percent}%")
print(highest_gc_samples.to_string(index=False))
print(f"\nLowest GC%: {min_gc_percent}%")
print(lowest_gc_samples.to_string(index=False))

-- DNA Analysis Report --
Total Samples Analysed: 50

--- Invalid Samples Flagged ---
Sample_ID Species           DNA_Sequence Invalid_Chars
     S050 Unknown ATGCGTXACGATCGATGCTAGC             X

Samples containing invalid nucleotides: : 1

Species Comparison
           Sample_Count  Avg_Length  Avg_GC_Percent  Avg_AT_Percent
Species                                                            
Chicken               7       30.43           45.53           54.47
Frog                  7       30.57           46.73           53.27
FruitFly              7       30.43           45.56           54.44
Human                 7       30.57           48.60           51.40
Mouse                 7       30.43           46.93           53.07
Rat                   7       30.57           46.74           53.26
Unknown               1       22.00           50.00           45.45
Zebrafish             7       30.43           49.31           50.69

Highest GC%: 53.33%
Sample_ID   Species  Length   GC%
    